In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")

# Training Params
BATCH_SIZE = 64
EPOCHS = 30
LABEL_PERCENTAGE = 0.05 # Competing on 20% data

# --- 1. Load Raw Data (Reuse your logic) ---
def load_raw_data():
    print("--- Loading Raw Data ---")
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None

    X = np.load(RAW_PAYLOAD_PATH).astype('float32')
    df = pd.read_csv(LABELS_PATH)

    # Reshape for CNN: (N, 10, 784) -> (N, 7840, 1)
    X = X.reshape(X.shape[0], -1, 1)
    return X, df

# --- 2. Define Your Exact CNN Architecture ---
# ... (Imports and Data Loading same as above) ...

# --- Deep Packet Architecture (Lotfollahi et al.) ---
def get_deeppacket_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv1D(filters=200, kernel_size=5, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 2
    x = layers.Conv1D(filters=200, kernel_size=4, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Flatten()(x)

    # Fully Connected
    x = layers.Dense(200, activation='relu')(x)
    x = layers.Dropout(0.05)(x) # Deep Packet uses low dropout [cite: 811]

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ... (Run Experiment and Main logic identical to above, just call get_deeppacket_cnn) ...

# --- 3. Run Experiment ---
def run_experiment(X, y_labels, task_name):
    print(f"\n>>> Running Supervised Baseline: {task_name}")

    # Encode Labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_labels)
    y_cat = to_categorical(y_int)
    num_classes = len(le.classes_)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_cat, train_size=LABEL_PERCENTAGE, stratify=y_int, random_state=42
    )

    # Train from Scratch
    model = get_deeppacket_cnn(X.shape[1:], num_classes)
    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)

    # Evaluate
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = np.argmax(y_test, axis=1)

    f1 = f1_score(y_true, y_pred, average='macro')
    print(f"    {task_name} F1: {f1:.4f}")

def main():
    X, df = load_raw_data()
    if X is None: return

    # 1. Binary Task
    run_experiment(X, df['binary_type'], "Binary")

    # 2. Category Task (VPN Only)
    vpn_mask = df['binary_type'] == 'VPN'
    run_experiment(X[vpn_mask], df.loc[vpn_mask, 'category'], "VPN Category")

    # 3. App Task (Top 6)
    targets = ['Skype', 'BitTorrent', 'Hangout', 'Facebook', 'YouTube', 'Email']
    # Note: Using raw names from cnn_labels, assuming they match your filtering logic
    app_mask = df['application'].isin(targets) & (df['binary_type'] == 'VPN')
    run_experiment(X[app_mask], df.loc[app_mask, 'application'], "VPN Apps")

if __name__ == "__main__":
    main()

--- Loading Raw Data ---

>>> Running Supervised Baseline: Binary
    Binary F1: 0.8061

>>> Running Supervised Baseline: VPN Category
    VPN Category F1: 0.6780

>>> Running Supervised Baseline: VPN Apps
    VPN Apps F1: 0.7536


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")

# Training Params
BATCH_SIZE = 64
EPOCHS = 30
LABEL_PERCENTAGE = 0.10 # Competing on 20% data

# --- 1. Load Raw Data (Reuse your logic) ---
def load_raw_data():
    print("--- Loading Raw Data ---")
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None

    X = np.load(RAW_PAYLOAD_PATH).astype('float32')
    df = pd.read_csv(LABELS_PATH)

    # Reshape for CNN: (N, 10, 784) -> (N, 7840, 1)
    X = X.reshape(X.shape[0], -1, 1)
    return X, df

# --- 2. Define Your Exact CNN Architecture ---
# ... (Imports and Data Loading same as above) ...

# --- Deep Packet Architecture (Lotfollahi et al.) ---
def get_deeppacket_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv1D(filters=200, kernel_size=5, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 2
    x = layers.Conv1D(filters=200, kernel_size=4, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Flatten()(x)

    # Fully Connected
    x = layers.Dense(200, activation='relu')(x)
    x = layers.Dropout(0.05)(x) # Deep Packet uses low dropout [cite: 811]

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ... (Run Experiment and Main logic identical to above, just call get_deeppacket_cnn) ...

# --- 3. Run Experiment ---
def run_experiment(X, y_labels, task_name):
    print(f"\n>>> Running Supervised Baseline: {task_name}")

    # Encode Labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_labels)
    y_cat = to_categorical(y_int)
    num_classes = len(le.classes_)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_cat, train_size=LABEL_PERCENTAGE, stratify=y_int, random_state=42
    )

    # Train from Scratch
    model = get_deeppacket_cnn(X.shape[1:], num_classes)
    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)

    # Evaluate
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = np.argmax(y_test, axis=1)

    f1 = f1_score(y_true, y_pred, average='macro')
    print(f"    {task_name} F1: {f1:.4f}")

def main():
    X, df = load_raw_data()
    if X is None: return

    # 1. Binary Task
    run_experiment(X, df['binary_type'], "Binary")

    # 2. Category Task (VPN Only)
    vpn_mask = df['binary_type'] == 'VPN'
    run_experiment(X[vpn_mask], df.loc[vpn_mask, 'category'], "VPN Category")

    # 3. App Task (Top 6)
    targets = ['Skype', 'BitTorrent', 'Hangout', 'Facebook', 'YouTube', 'Email']
    # Note: Using raw names from cnn_labels, assuming they match your filtering logic
    app_mask = df['application'].isin(targets) & (df['binary_type'] == 'VPN')
    run_experiment(X[app_mask], df.loc[app_mask, 'application'], "VPN Apps")

if __name__ == "__main__":
    main()

--- Loading Raw Data ---

>>> Running Supervised Baseline: Binary
    Binary F1: 0.8499

>>> Running Supervised Baseline: VPN Category
    VPN Category F1: 0.7577

>>> Running Supervised Baseline: VPN Apps
    VPN Apps F1: 0.8344


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")

# Training Params
BATCH_SIZE = 64
EPOCHS = 30
LABEL_PERCENTAGE = 0.20 # Competing on 20% data

# --- 1. Load Raw Data (Reuse your logic) ---
def load_raw_data():
    print("--- Loading Raw Data ---")
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None

    X = np.load(RAW_PAYLOAD_PATH).astype('float32')
    df = pd.read_csv(LABELS_PATH)

    # Reshape for CNN: (N, 10, 784) -> (N, 7840, 1)
    X = X.reshape(X.shape[0], -1, 1)
    return X, df

# --- 2. Define Your Exact CNN Architecture ---
# ... (Imports and Data Loading same as above) ...

# --- Deep Packet Architecture (Lotfollahi et al.) ---
def get_deeppacket_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv1D(filters=200, kernel_size=5, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 2
    x = layers.Conv1D(filters=200, kernel_size=4, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Flatten()(x)

    # Fully Connected
    x = layers.Dense(200, activation='relu')(x)
    x = layers.Dropout(0.05)(x) # Deep Packet uses low dropout [cite: 811]

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ... (Run Experiment and Main logic identical to above, just call get_deeppacket_cnn) ...

# --- 3. Run Experiment ---
def run_experiment(X, y_labels, task_name):
    print(f"\n>>> Running Supervised Baseline: {task_name}")

    # Encode Labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_labels)
    y_cat = to_categorical(y_int)
    num_classes = len(le.classes_)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_cat, train_size=LABEL_PERCENTAGE, stratify=y_int, random_state=42
    )

    # Train from Scratch
    model = get_deeppacket_cnn(X.shape[1:], num_classes)
    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)

    # Evaluate
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = np.argmax(y_test, axis=1)

    f1 = f1_score(y_true, y_pred, average='macro')
    print(f"    {task_name} F1: {f1:.4f}")

def main():
    X, df = load_raw_data()
    if X is None: return

    # 1. Binary Task
    run_experiment(X, df['binary_type'], "Binary")

    # 2. Category Task (VPN Only)
    vpn_mask = df['binary_type'] == 'VPN'
    run_experiment(X[vpn_mask], df.loc[vpn_mask, 'category'], "VPN Category")

    # 3. App Task (Top 6)
    targets = ['Skype', 'BitTorrent', 'Hangout', 'Facebook', 'YouTube', 'Email']
    # Note: Using raw names from cnn_labels, assuming they match your filtering logic
    app_mask = df['application'].isin(targets) & (df['binary_type'] == 'VPN')
    run_experiment(X[app_mask], df.loc[app_mask, 'application'], "VPN Apps")

if __name__ == "__main__":
    main()

--- Loading Raw Data ---

>>> Running Supervised Baseline: Binary
    Binary F1: 0.8856

>>> Running Supervised Baseline: VPN Category
    VPN Category F1: 0.7861

>>> Running Supervised Baseline: VPN Apps
    VPN Apps F1: 0.8646


In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
RAW_PAYLOAD_PATH = os.path.join(BASE_PATH, "cnn_payload_data.npy")
LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")

# Training Params
BATCH_SIZE = 64
EPOCHS = 30
LABEL_PERCENTAGE = 0.30 # Competing on 20% data

# --- 1. Load Raw Data (Reuse your logic) ---
def load_raw_data():
    print("--- Loading Raw Data ---")
    if not os.path.exists(RAW_PAYLOAD_PATH): return None, None

    X = np.load(RAW_PAYLOAD_PATH).astype('float32')
    df = pd.read_csv(LABELS_PATH)

    # Reshape for CNN: (N, 10, 784) -> (N, 7840, 1)
    X = X.reshape(X.shape[0], -1, 1)
    return X, df

# --- 2. Define Your Exact CNN Architecture ---
# ... (Imports and Data Loading same as above) ...

# --- Deep Packet Architecture (Lotfollahi et al.) ---
def get_deeppacket_cnn(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv1D(filters=200, kernel_size=5, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 2
    x = layers.Conv1D(filters=200, kernel_size=4, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Flatten()(x)

    # Fully Connected
    x = layers.Dense(200, activation='relu')(x)
    x = layers.Dropout(0.05)(x) # Deep Packet uses low dropout [cite: 811]

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ... (Run Experiment and Main logic identical to above, just call get_deeppacket_cnn) ...

# --- 3. Run Experiment ---
def run_experiment(X, y_labels, task_name):
    print(f"\n>>> Running Supervised Baseline: {task_name}")

    # Encode Labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_labels)
    y_cat = to_categorical(y_int)
    num_classes = len(le.classes_)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_cat, train_size=LABEL_PERCENTAGE, stratify=y_int, random_state=42
    )

    # Train from Scratch
    model = get_deeppacket_cnn(X.shape[1:], num_classes)
    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)

    # Evaluate
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = np.argmax(y_test, axis=1)

    f1 = f1_score(y_true, y_pred, average='macro')
    print(f"    {task_name} F1: {f1:.4f}")

def main():
    X, df = load_raw_data()
    if X is None: return

    # 1. Binary Task
    run_experiment(X, df['binary_type'], "Binary")

    # 2. Category Task (VPN Only)
    vpn_mask = df['binary_type'] == 'VPN'
    run_experiment(X[vpn_mask], df.loc[vpn_mask, 'category'], "VPN Category")

    # 3. App Task (Top 6)
    targets = ['Skype', 'BitTorrent', 'Hangout', 'Facebook', 'YouTube', 'Email']
    # Note: Using raw names from cnn_labels, assuming they match your filtering logic
    app_mask = df['application'].isin(targets) & (df['binary_type'] == 'VPN')
    run_experiment(X[app_mask], df.loc[app_mask, 'application'], "VPN Apps")

if __name__ == "__main__":
    main()

--- Loading Raw Data ---

>>> Running Supervised Baseline: Binary
    Binary F1: 0.8918

>>> Running Supervised Baseline: VPN Category
    VPN Category F1: 0.8319

>>> Running Supervised Baseline: VPN Apps
    VPN Apps F1: 0.8738
